In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time


In [5]:
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")

driver = webdriver.Chrome(options=options)
driver.get("https://ffp.nl/vind-een-planner/?gespecialiseerd_vermogensopbouw=vermogensopbouw&pageNumber=1")

links = []

while True:
    time.sleep(2)
    links.append(driver.current_url)
    try:
        next_link = driver.find_element("xpath", '//a[contains(@class, "back-next next")]').get_attribute("href")
        if not next_link:
            break
        driver.get(next_link)
    except:
        break

driver.quit()


In [6]:
options = Options()
options.add_argument("--headless=new")
driver = webdriver.Chrome(options=options)

profile_links = []

for link in links:
    driver.get(link)
    divs = driver.find_elements("xpath", '//div[@class="inner-block"]')
    for div in divs:
        onclick_value = div.get_attribute("onclick")
        if onclick_value:
            relative_link = onclick_value.split("location.href=")[1].strip("';")
            profile_links.append("https://ffp.nl" + relative_link)

driver.quit()


In [7]:
len(profile_links)

913

In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
import pandas as pd
import time
options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)
data = []

for link in profile_links:
    driver.get(link)
    time.sleep(2)  
    name = phone = email = website = None
    try:
        contact_div = driver.find_element("xpath", '//div[@class="btn-wrap"]')
    except NoSuchElementException:
        contact_div = None
    try:
        phone_element = driver.find_element("xpath", '//a[contains(@href, "tel:")]')
        phone = phone_element.get_attribute("href").replace("tel:", "")
    except NoSuchElementException:
        phone = None
    try:
        email_element = contact_div.find_element("xpath", './/a[starts-with(@href, "mailto:")]') if contact_div else None
        email = email_element.get_attribute("href").replace("mailto:", "") if email_element else None
    except NoSuchElementException:
        email = None
    try:
        website_element = contact_div.find_element("xpath", './/a[starts-with(@href, "http")]') if contact_div else None
        website = website_element.get_attribute("href") if website_element else None
    except NoSuchElementException:
        website = None
    try:
        name_element = driver.find_element("xpath", '//div[@class="default-content"]/h1')
        name = name_element.text.strip()
    except NoSuchElementException:
        name = None
    data.append({
        "Name": name,
        "Phone": phone,
        "Email": email,
        "Website": website,
        "Profile Link": link
    })
driver.quit()
df = pd.DataFrame(data)

In [9]:
df

,Name,Phone,Email,Website,Profile Link
0,Mevrouw A. Ivanovic CFP® B.ec,None,Anita.Ivanovic@nl.abnamro.com,https://www.abnamro.nl/nl/privatebanking/index...,https://ffp.nl/planner/aivanovicbec
1,De heer G.J. Krol CFP,076-5310140,gertjan@withequip.com,None,https://ffp.nl/planner/gjkrolcfp
2,De heer J. Corts CFP® DSI,None,jasper.corts@nl.abnamro.com,https://www.abnamro.com/,https://ffp.nl/planner/jcortsdsi
3,De heer M. May CFP,06-30037264,martijn.may@ing.com,None,https://ffp.nl/planner/mmaycfp
4,De heer N. Hajjari CFP® DSI,0800-1737,n.hajjari@vanlanschotkempen.com,https://www.vanlanschotkempen.com/nl-nl,https://ffp.nl/planner/nhajjaridsi
...,...,...,...,...,...
908,De heer J.H.J. Brauwers CFP®,06-21208831,vtboa@xs4all.nl,None,https://ffp.nl/planner/jhjbrauwers
909,De heer M.M.B. Brants CFP®,06 - 8205 98 66,marcel.brants@sns.nl,None,https://ffp.nl/planner/mmbbrants
910,De heer M. de Bree CFP®,0161 43 15 99,m.debree@accountenzbreda.nl,http://www.accountenzbreda.nl/,https://ffp.nl/planner/mdebree
911,De heer N. Aarts CFP®,0620595796,niels.aarts@nl.abnamro.com,http://www.abnamro.nl/,https://ffp.nl/planner/naarts


In [15]:
df.to_csv(r"C:\Users\ABC\OneDrive - Notley Green Primary School\Documents\data Analytics completed projects\Projects\profile-data\Profile Data.csv", index=False)